# LiDAR Ground Segmentation + Slope/Curvature Pipeline

End-to-end pipeline in one notebook. For each timestamp in `TIMESTAMPS`:

**Stage 1 — Ground segmentation** (two parallel filters on the same MCAP frame):

| Filter | Output file | Notes |
|---|---|---|
| **Patchwork++** | `{timestamp}_patchwork.bin` | `/tf_static` applied *after* segmentation |
| **CSF**         | `{timestamp}_csf.bin`       | hardcoded sensor transforms applied *before* segmentation |

Both files are nuScenes-format `(N, 5)` `float32` so Stage 2 reads them with one loader.

**Stage 2 — Slope & curvature** (run on *both* `.bin` files):

For each `.bin` the four methods (PCA, RANSAC, height-derivative, IRLS quadric) produce
a forward 1-D road profile. Each profile is saved as `{method}_{filter}_{timestamp}.npz`
— eight `.npz` files per timestamp in total.

> Only the MCAP workflow is supported. All visualisations, before/after plots, heatmaps,
> the GLE post-filter, and the `.ply` / nuScenes-`.bin` / KITTI loaders from the original
> notebooks have been removed. Calculation methods are unchanged.

## 1. Configuration — single source of truth

All paths, timestamps and tunable constants for the whole pipeline live here.
Nothing else in the notebook needs editing.

In [ ]:
# ==========================================================
# INPUT — MCAP bag
# ==========================================================
MCAP_PATH = r"D:\Data_gathered\2026_05_22\Rosbag\10_40_00\rosbag\rosbag_0.mcap"

# Timestamps (Unix seconds) to process. One file per filter per timestamp.
TIMESTAMPS = [
    1779439338.625917639,
    # 1779439340.000000000,
    # add more as you like — bag is indexed once and reused
]

# ==========================================================
# OUTPUT directories
# ==========================================================
GROUND_BIN_DIR = r"D:\Filtered bins"
RESULTS_DIR    = r"D:\Validation_results"   # slope/curvature .npz files go here

# ==========================================================
# LiDAR topics (MCAP) and base frame
# ==========================================================
LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]
BASE_FRAME = "base_link"

# ==========================================================
# Patchwork++ parameters (SenseBike)
# ==========================================================
PW_SENSOR_HEIGHT = 0.90   # [m] LiDAR origin height above ground
PW_MIN_RANGE     = 1.0    # [m]
PW_MAX_RANGE     = 50.0   # [m]
PW_VERBOSE       = False

# ==========================================================
# CSF parameters (cloth simulation filter)
# ==========================================================
CSF_CLOTH_RESOLUTION = 0.3   # [m] cloth grid size
CSF_SLOPE_SMOOTH     = True  # True = gentle slopes
CSF_THRESHOLD        = 0.1   # [m] max distance from cloth to count as ground

# Crop applied BEFORE CSF (base_link frame)
CSF_X_RANGE = 20.0   # ±X [m]
CSF_Y_RANGE = 1.0    # ±Y [m]

# Sensor transforms used by CSF (hardcoded, read from /tf_static via Foxglove).
# Applied per-topic BEFORE merging and BEFORE CSF.
SENSOR_TRANSFORMS = {
    "/rslidar/M1P_deskewed": {"translation": [ 0.800,  0.000, 0.876], "rotation_rpy_deg": [0.1, -1.7,   1.7]},
    "/rslidar/helios_R":     {"translation": [-0.743, -0.243, 0.857], "rotation_rpy_deg": [2.4, -2.6, -179.9]},
    "/rslidar/helios_L":     {"translation": [-0.745,  0.191, 0.876], "rotation_rpy_deg": [3.5, -2.1,  179.9]},
}

# ==========================================================
# Slope / curvature parameters (Stage 2)
# ==========================================================
ROI_X_RANGE = (-5.0, 20.0)    # forward strip used for 1-D profile [m]
ROI_Y_RANGE = (-1.0,  1.0)    # lateral half-width [m]

PROFILE_BIN_SIZE      = 0.5   # [m] forward bin width
PROFILE_MIN_PTS       = 15    # min points to attempt a plane fit
PROFILE_MEDIAN_WINDOW = 5     # outlier suppression
PROFILE_SMOOTH_WINDOW = 7     # smoothing after the median filter

# RANSAC
RANSAC_N_ITER      = 50
RANSAC_DIST_THRESH = 0.05   # [m]
RANSAC_MIN_INLIERS = 8
RANSAC_SEED        = 42

# IRLS quadric (global robust surface fit)
IRLS_MAX_ITER  = 30
IRLS_TOL       = 1e-7
IRLS_C_TUKEY   = 4.685    # Tukey biweight constant (95% efficiency at Gaussian)
IRLS_Y_CENTER  = 0.0      # y of the slice the 1-D profile is read from [m]

print("Configuration loaded.")
print(f"  MCAP:        {MCAP_PATH}")
print(f"  Timestamps:  {len(TIMESTAMPS)}")
print(f"  Bin dir:     {GROUND_BIN_DIR}")
print(f"  Results dir: {RESULTS_DIR}")


In [ ]:
import os
import numpy as np
from pathlib import Path

np.set_printoptions(suppress=True, precision=4)

os.makedirs(GROUND_BIN_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)


## 2. Shared MCAP loaders

The bag index and `/tf_static` tree are built once (cached) and reused across both
filters and all timestamps, so adding more timestamps costs only the windowed read
+ segmentation + save.

In [ ]:
# ----------------------------------------------------------
# PointCloud2 -> (xyz, frame_id) extractor
# ----------------------------------------------------------
_PF_DTYPE = {1: ("i1", 1), 2: ("u1", 1), 3: ("i2", 2), 4: ("u2", 2),
             5: ("i4", 4), 6: ("u4", 4), 7: ("f4", 4), 8: ("f8", 8)}


def _pointcloud2_to_xyz(msg):
    field_map = {f.name: f for f in msg.fields}
    missing = [k for k in ("x", "y", "z") if k not in field_map]
    if missing:
        raise RuntimeError(f"PointCloud2 missing {missing}. Have: {list(field_map)}")
    n = msg.width * msg.height
    raw = np.frombuffer(msg.data, dtype=np.uint8).reshape(n, msg.point_step)
    xyz = np.empty((n, 3), dtype=np.float64)
    for i, name in enumerate(("x", "y", "z")):
        f = field_map[name]
        dtype_str, size = _PF_DTYPE[f.datatype]
        col_bytes = raw[:, f.offset:f.offset + size].copy()
        xyz[:, i] = col_bytes.view(np.dtype(dtype_str)).flatten().astype(np.float64)
    valid = np.isfinite(xyz).all(axis=1)
    return xyz[valid], msg.header.frame_id


# ----------------------------------------------------------
# Bag index (cached)
# ----------------------------------------------------------
_MCAP_INDEX_CACHE = {}


def _build_mcap_index(path):
    from rosbags.highlevel import AnyReader
    if path in _MCAP_INDEX_CACHE:
        return _MCAP_INDEX_CACHE[path]
    print("[mcap] Indexing bag (one-time)... ", end="", flush=True)
    index = {}
    with AnyReader([Path(path)]) as reader:
        pc2_conns = [c for c in reader.connections
                     if c.msgtype == "sensor_msgs/msg/PointCloud2"]
        for c in pc2_conns:
            index[c.topic] = []
        for conn, ts, raw in reader.messages(connections=pc2_conns):
            index[conn.topic].append(ts)
    for topic in index:
        index[topic] = np.asarray(index[topic], dtype=np.int64)
    _MCAP_INDEX_CACHE[path] = {"index": index}
    print(f"done. Topics: {list(index.keys())}")
    return _MCAP_INDEX_CACHE[path]


# ----------------------------------------------------------
# /tf_static tree (cached)
# ----------------------------------------------------------
_TF_TREE_CACHE = {}


def _quat_to_R(x, y, z, w):
    n = x * x + y * y + z * z + w * w
    if n < 1e-12:
        return np.eye(3)
    s = 2.0 / n
    return np.array([
        [1 - s*(y*y + z*z),     s*(x*y - z*w),     s*(x*z + y*w)],
        [    s*(x*y + z*w), 1 - s*(x*x + z*z),     s*(y*z - x*w)],
        [    s*(x*z - y*w),     s*(y*z + x*w), 1 - s*(x*x + y*y)],
    ])


def _build_tf_tree(mcap_path):
    from rosbags.highlevel import AnyReader
    if mcap_path in _TF_TREE_CACHE:
        return _TF_TREE_CACHE[mcap_path]
    tree = {}
    with AnyReader([Path(mcap_path)]) as reader:
        wanted = [c for c in reader.connections if c.topic == "/tf_static"]
        if not wanted:
            print("[tf] WARNING: /tf_static not found -- no calibration available.")
            _TF_TREE_CACHE[mcap_path] = tree
            return tree
        for conn, ts, raw in reader.messages(connections=wanted):
            msg = reader.deserialize(raw, conn.msgtype)
            for tf in msg.transforms:
                parent = tf.header.frame_id
                child  = tf.child_frame_id
                t = np.array([tf.transform.translation.x,
                              tf.transform.translation.y,
                              tf.transform.translation.z])
                q = tf.transform.rotation
                R = _quat_to_R(q.x, q.y, q.z, q.w)
                tree[child] = (parent, R, t)
    print(f"[tf] Parsed {len(tree)} static transforms.")
    _TF_TREE_CACHE[mcap_path] = tree
    return tree


def _resolve_to_base(tree, child_frame, base_frame="base_link", max_depth=10):
    R_acc = np.eye(3); t_acc = np.zeros(3); cur = child_frame
    for _ in range(max_depth):
        if cur == base_frame:
            return R_acc, t_acc
        if cur not in tree:
            raise RuntimeError(f"'{cur}' not in /tf_static -- cannot reach '{base_frame}'.")
        parent, R, t = tree[cur]
        t_acc = R @ t_acc + t
        R_acc = R @ R_acc
        cur = parent
    raise RuntimeError(f"TF chain too deep from '{child_frame}'.")


def _apply_tf(xyz, R, t):
    return xyz @ R.T + t


# ----------------------------------------------------------
# Windowed multi-topic read: returns per-topic dict {topic: (xyz, frame_id)}
# ----------------------------------------------------------
def load_mcap_frame_per_topic(path, topics, target_sec, tol_sec=0.05):
    """Pick the message in each topic nearest `target_sec`. The reference
    timestamp (for matching) is the first topic's nearest message."""
    from rosbags.highlevel import AnyReader

    cache = _build_mcap_index(path)
    ref_topic = topics[0]
    ts_list = cache["index"][ref_topic]
    target_ns = int(target_sec * 1e9)
    ref_idx = int(np.argmin(np.abs(ts_list - target_ns)))
    target_ns = int(ts_list[ref_idx])
    diff_s = abs(ts_list[ref_idx] - int(target_sec * 1e9)) / 1e9
    print(f"  [mcap] Timestamp {target_sec:.3f}s -> frame {ref_idx} on {ref_topic} "
          f"(off by {diff_s*1000:.1f} ms)")

    tol_ns = int(tol_sec * 1e9)
    start_ns = target_ns - tol_ns
    end_ns   = target_ns + tol_ns
    out = {t: None for t in topics}

    with AnyReader([Path(path)]) as reader:
        wanted = [c for c in reader.connections if c.topic in set(topics)]
        for conn, ts, raw in reader.messages(connections=wanted,
                                             start=start_ns, stop=end_ns + 1):
            if out[conn.topic] is None:
                msg = reader.deserialize(raw, conn.msgtype)
                xyz, fid = _pointcloud2_to_xyz(msg)
                out[conn.topic] = (xyz, fid)
                if all(v is not None for v in out.values()):
                    break

    for t in topics:
        if out[t] is None:
            print(f"  [warn] {t}: no message within +/-{tol_sec*1000:.0f} ms")
        else:
            xyz, fid = out[t]
            print(f"  {t}: {len(xyz):,} pts  (frame: {fid})")
    return out


## 3. Patchwork++ pipeline

Per topic: run Patchwork++ in the sensor's own frame, *then* apply `/tf_static` to
move the surviving ground points into `base_link`. (Calibrating first would shift
Z values and break the `sensor_height` prior.)

In [ ]:
import pypatchworkpp


def _build_patchworkpp():
    params = pypatchworkpp.Parameters()
    params.sensor_height = float(PW_SENSOR_HEIGHT)
    params.min_range     = float(PW_MIN_RANGE)
    params.max_range     = float(PW_MAX_RANGE)
    params.verbose       = bool(PW_VERBOSE)
    return pypatchworkpp.patchworkpp(params)


PatchworkPP = _build_patchworkpp()
print(f"Patchwork++ ready: sensor_height={PW_SENSOR_HEIGHT} m, "
      f"range [{PW_MIN_RANGE}, {PW_MAX_RANGE}] m")


def _pw_segment(points_xyz):
    """Run Patchwork++ on (N, 3). Returns the ground subset (M, 3)."""
    pts4 = np.hstack([points_xyz, np.zeros((len(points_xyz), 1))]).astype(np.float32)
    PatchworkPP.estimateGround(pts4)
    return np.asarray(PatchworkPP.getGround())[:, :3]


def run_patchwork(per_topic):
    """Patchwork++ per topic in sensor frame, then /tf_static into base_link.
    `per_topic`: {topic: (xyz, frame_id)} from load_mcap_frame_per_topic.
    Returns ground points (N, 3) in base_link."""
    tree = _build_tf_tree(MCAP_PATH)
    have_tf = bool(tree)
    g_parts = []
    for topic, payload in per_topic.items():
        if payload is None:
            continue
        raw_xyz, frame_id = payload
        g = _pw_segment(raw_xyz)
        if have_tf:
            try:
                R, t = _resolve_to_base(tree, frame_id, BASE_FRAME)
                g = _apply_tf(g, R, t)
            except RuntimeError as e:
                print(f"    [warn] {topic}: {e} -- leaving uncalibrated")
        g_parts.append(g)
    if not g_parts:
        raise RuntimeError("Patchwork++: no topics produced ground points.")
    return np.concatenate(g_parts, axis=0)


## 4. CSF pipeline

CSF works on the *merged* cloud, not per sensor. So: calibrate each topic with the
hardcoded `SENSOR_TRANSFORMS` first → merge into base_link → crop to the strip in
front of the bike → run CSF on the cropped cloud.

In [ ]:
import CSF


def _rpy_deg_to_R(roll_deg, pitch_deg, yaw_deg):
    r, p, y = np.radians(roll_deg), np.radians(pitch_deg), np.radians(yaw_deg)
    Rx = np.array([[1, 0, 0], [0, np.cos(r), -np.sin(r)], [0, np.sin(r), np.cos(r)]])
    Ry = np.array([[np.cos(p), 0, np.sin(p)], [0, 1, 0], [-np.sin(p), 0, np.cos(p)]])
    Rz = np.array([[np.cos(y), -np.sin(y), 0], [np.sin(y), np.cos(y), 0], [0, 0, 1]])
    return Rz @ Ry @ Rx


def _transform_points(points, translation, rotation_rpy_deg):
    R = _rpy_deg_to_R(*rotation_rpy_deg)
    t = np.array(translation, dtype=np.float64)
    return points @ R.T + t


def run_csf(per_topic):
    """Calibrate per-topic with SENSOR_TRANSFORMS, merge, crop, run CSF.
    Returns ground points (N, 3) in base_link."""
    parts = []
    for topic, payload in per_topic.items():
        if payload is None or topic not in SENSOR_TRANSFORMS:
            continue
        raw_xyz, _frame_id = payload
        tf = SENSOR_TRANSFORMS[topic]
        parts.append(_transform_points(raw_xyz, tf["translation"], tf["rotation_rpy_deg"]))
    if not parts:
        raise RuntimeError("CSF: no topics with matching SENSOR_TRANSFORMS.")
    merged = np.vstack(parts)

    # Crop FIRST, then run CSF on the cropped data (same as v9 CSF notebook)
    mask = (np.abs(merged[:, 0]) <= CSF_X_RANGE) & (np.abs(merged[:, 1]) <= CSF_Y_RANGE)
    merged = merged[mask]
    if len(merged) < 10:
        raise RuntimeError(f"CSF: only {len(merged)} points survived the crop.")

    csf = CSF.CSF()
    csf.params.bSloopSmooth     = CSF_SLOPE_SMOOTH
    csf.params.cloth_resolution = CSF_CLOTH_RESOLUTION
    csf.params.threshold        = CSF_THRESHOLD
    csf.setPointCloud(merged)

    ground_idx, non_ground_idx = CSF.VecInt(), CSF.VecInt()
    csf.do_filtering(ground_idx, non_ground_idx)
    return merged[np.array(list(ground_idx))]


## 5. Saving — nuScenes `(N, 5)` `float32`

Both filter outputs are written in the same format so Stage 2 can use one loader.
The timestamp encoded in the filename is the *requested* one (integer Unix
milliseconds), so it matches exactly what was put in `TIMESTAMPS`.

In [ ]:
def timestamp_stem(ts_sec):
    """Unix-ms integer stem: 1779439338.625917639 -> '1779439338625'."""
    return str(int(round(ts_sec * 1000)))


def save_as_nuscenes_bin(points_xyz, path):
    """Save (N, 3) XYZ as nuScenes-format (N, 5) float32 with zero intensity/ring."""
    n = len(points_xyz)
    arr = np.zeros((n, 5), dtype=np.float32)
    arr[:, :3] = points_xyz.astype(np.float32)
    arr.tofile(path)
    print(f"    saved {n:,} pts -> {os.path.basename(path)}  ({arr.nbytes / 1024:.1f} KB)")


## 6. Stage 1 driver — both filters, every timestamp

For each timestamp we read the per-topic clouds *once*, then hand the same dict to
both filters. The bag index and TF tree are reused across the whole loop.

In [ ]:
STAGE1_OUTPUTS = []   # list of (ts_sec, patchwork_path, csf_path)

for k, ts in enumerate(TIMESTAMPS, start=1):
    print("=" * 64)
    print(f"[{k}/{len(TIMESTAMPS)}]  Stage 1  --  t = {ts:.6f} s")
    print("=" * 64)

    per_topic = load_mcap_frame_per_topic(MCAP_PATH, LIDAR_TOPICS, target_sec=ts)
    stem = timestamp_stem(ts)

    # --- Patchwork++ ---
    print("  [Patchwork++]")
    ground_pw = run_patchwork(per_topic)
    pw_path = os.path.join(GROUND_BIN_DIR, f"{stem}_patchwork.bin")
    save_as_nuscenes_bin(ground_pw, pw_path)

    # --- CSF ---
    print("  [CSF]")
    ground_csf = run_csf(per_topic)
    csf_path = os.path.join(GROUND_BIN_DIR, f"{stem}_csf.bin")
    save_as_nuscenes_bin(ground_csf, csf_path)

    STAGE1_OUTPUTS.append((ts, pw_path, csf_path))

print(f"\nStage 1 complete: {len(STAGE1_OUTPUTS)} timestamp(s), {2 * len(STAGE1_OUTPUTS)} .bin file(s).")


## 7. Stage 2 — Slope & curvature

The four methods share everything except how slope and curvature are obtained:

| Method | Fitting unit | Curvature comes from |
|---|---|---|
| **PCA**             | covariance → eigenvectors (per bin)        | derivative of slope (`-n_x / n_z`) |
| **RANSAC**          | random samples → consensus plane (per bin) | derivative of slope (`-n_x / n_z`) |
| **Height-derivative** | median z per bin (no plane fit)          | `np.gradient` applied twice to z |
| **IRLS quadric**    | one global $z=ax^2+bxy+cy^2+dx+ey+f$ fit with Tukey biweight | analytic derivatives of the quadric along the slice |

The first three are *local* per-bin estimators; IRLS is a *global* surface fit that
trades local responsiveness for noise robustness and produces smooth analytic
derivatives along the strip slice at `y = IRLS_Y_CENTER`. The two smoothers
(median then moving-average) and the differentiation step still apply to PCA, RANSAC
and height-derivative; IRLS skips them because the quadric is already smooth.
Calculation methods are unchanged from `compute_slope_curvature_v2.ipynb` and
`curvature_via_irls_quadric_1.ipynb`.

In [ ]:
def load_nuscenes_bin(path):
    """(N, 5) float32 -> (N, 3) float64 XYZ."""
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)
    return pts[:, :3].astype(np.float64)


# ----------------------------------------------------------
# Smoothing
# ----------------------------------------------------------
def _median_filter(y, window):
    if window <= 1:
        return y.copy()
    pad = window // 2
    y_pad = np.pad(y, pad, mode="reflect")
    out = np.empty_like(y)
    for i in range(len(y)):
        out[i] = np.median(y_pad[i:i + window])
    return out


def _moving_average(y, window):
    if window <= 1:
        return y.copy()
    pad = window // 2
    y_pad = np.pad(y, pad, mode="reflect")
    kernel = np.ones(window) / window
    return np.convolve(y_pad, kernel, mode="valid")[:len(y)]


def _two_stage_smooth(y, median_window, smooth_window):
    return _moving_average(_median_filter(y, median_window), smooth_window)


In [ ]:
# ----------------------------------------------------------
# Forward 1-D profile -- slope from a plane normal (PCA & RANSAC)
# ----------------------------------------------------------
def compute_road_profile_from_normals(ground_points, fit_normal, method_label):
    x_range       = ROI_X_RANGE
    y_half_width  = abs(ROI_Y_RANGE[1])
    bin_size      = PROFILE_BIN_SIZE
    min_pts       = PROFILE_MIN_PTS
    median_window = PROFILE_MEDIAN_WINDOW
    smooth_window = PROFILE_SMOOTH_WINDOW

    m = ((ground_points[:, 0] >= x_range[0]) &
         (ground_points[:, 0] <= x_range[1]) &
         (np.abs(ground_points[:, 1]) <= y_half_width))
    pts = ground_points[m]
    if len(pts) < 10:
        print(f"  [{method_label}] Not enough ground points in the forward strip.")
        return None

    x_edges   = np.arange(x_range[0], x_range[1] + bin_size, bin_size)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    n_bins    = len(x_centers)

    z_bin    = np.full(n_bins, np.nan)
    dzdx_bin = np.full(n_bins, np.nan)

    for i in range(n_bins):
        in_bin = (pts[:, 0] >= x_edges[i]) & (pts[:, 0] < x_edges[i + 1])
        bin_pts = pts[in_bin]
        if len(bin_pts) < min_pts:
            continue
        z_bin[i] = float(np.median(bin_pts[:, 2]))
        try:
            n = fit_normal(bin_pts)
            if abs(n[2]) > 1e-6:
                dzdx_bin[i] = -n[0] / n[2]
        except Exception:
            pass

    valid_z = ~np.isnan(z_bin)
    valid_s = ~np.isnan(dzdx_bin)
    if valid_z.sum() < 5 or valid_s.sum() < 5:
        print(f"  [{method_label}] Not enough populated bins.")
        return None

    z_filled    = np.interp(x_centers, x_centers[valid_z], z_bin[valid_z])
    dzdx_filled = np.interp(x_centers, x_centers[valid_s], dzdx_bin[valid_s])

    z_smooth    = _two_stage_smooth(z_filled,    median_window, smooth_window)
    dzdx_smooth = _two_stage_smooth(dzdx_filled, median_window, smooth_window)

    d2zdx2 = np.gradient(dzdx_smooth, bin_size)
    kappa  = d2zdx2 / (1.0 + dzdx_smooth ** 2) ** 1.5

    return dict(
        x=x_centers,
        z=z_smooth,
        z_raw=z_bin,
        slope_deg=np.degrees(np.arctan(dzdx_smooth)),
        slope_raw_deg=np.degrees(np.arctan(dzdx_bin)),
        kappa=kappa,
        kappa_abs=np.abs(kappa),
        valid_z_mask=valid_z,
        valid_s_mask=valid_s,
        bin_size=bin_size,
        method=method_label,
    )


# ----------------------------------------------------------
# Method 1: PCA per bin
# ----------------------------------------------------------
def fit_pca_normal(bin_pts):
    pts_c = bin_pts.astype(np.float64) - bin_pts.mean(axis=0)
    C = (pts_c.T @ pts_c) / max(len(pts_c) - 1, 1)
    eigvals, eigvecs = np.linalg.eigh(C)
    n = eigvecs[:, 0]
    if n[2] < 0:
        n = -n
    return n


# ----------------------------------------------------------
# Method 2: RANSAC per bin
# ----------------------------------------------------------
def fit_ransac_normal(bin_pts,
                      n_iter=RANSAC_N_ITER,
                      dist_thresh=RANSAC_DIST_THRESH,
                      min_inliers=RANSAC_MIN_INLIERS,
                      rng=None):
    _ransac_rng = np.random.default_rng(RANSAC_SEED)
    rng = rng or _ransac_rng
    pts = bin_pts.astype(np.float64)
    n_pts = len(pts)
    if n_pts < 3:
        raise RuntimeError(f"RANSAC needs >=3 pts, got {n_pts}.")

    best_normal = None
    best_count  = -1
    for _ in range(n_iter):
        idx = rng.choice(n_pts, 3, replace=False)
        p0, p1, p2 = pts[idx]
        n_try = np.cross(p1 - p0, p2 - p0)
        norm = np.linalg.norm(n_try)
        if norm < 1e-9:
            continue
        n_try = n_try / norm
        if n_try[2] < 0:
            n_try = -n_try
        n_inliers = int((np.abs((pts - p0) @ n_try) < dist_thresh).sum())
        if n_inliers > best_count:
            best_count  = n_inliers
            best_normal = n_try

    if best_normal is None or best_count < min_inliers:
        raise RuntimeError(
            f"RANSAC failed: best consensus only {best_count} inliers "
            f"(need {min_inliers}, dist_thresh={dist_thresh} m, "
            f"n_iter={n_iter}, bin has {n_pts} pts)."
        )
    return best_normal


# ----------------------------------------------------------
# Method 3: Height derivative (no plane fit)
# ----------------------------------------------------------
def compute_road_profile_from_heights(ground_points, method_label="height-deriv"):
    x_range       = ROI_X_RANGE
    y_half_width  = abs(ROI_Y_RANGE[1])
    bin_size      = PROFILE_BIN_SIZE
    min_pts       = PROFILE_MIN_PTS
    median_window = PROFILE_MEDIAN_WINDOW
    smooth_window = PROFILE_SMOOTH_WINDOW

    m = ((ground_points[:, 0] >= x_range[0]) &
         (ground_points[:, 0] <= x_range[1]) &
         (np.abs(ground_points[:, 1]) <= y_half_width))
    pts = ground_points[m]
    if len(pts) < 10:
        print(f"  [{method_label}] Not enough ground points in the forward strip.")
        return None

    x_edges   = np.arange(x_range[0], x_range[1] + bin_size, bin_size)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    n_bins    = len(x_centers)

    z_bin = np.full(n_bins, np.nan)
    for i in range(n_bins):
        in_bin = (pts[:, 0] >= x_edges[i]) & (pts[:, 0] < x_edges[i + 1])
        bin_pts = pts[in_bin]
        if len(bin_pts) < min_pts:
            continue
        z_bin[i] = float(np.median(bin_pts[:, 2]))

    valid_z = ~np.isnan(z_bin)
    if valid_z.sum() < 5:
        print(f"  [{method_label}] Not enough populated bins.")
        return None

    z_filled    = np.interp(x_centers, x_centers[valid_z], z_bin[valid_z])
    z_smooth    = _two_stage_smooth(z_filled, median_window, smooth_window)
    dzdx_smooth = np.gradient(z_smooth,    bin_size)
    d2zdx2      = np.gradient(dzdx_smooth, bin_size)
    kappa       = d2zdx2 / (1.0 + dzdx_smooth ** 2) ** 1.5

    dzdx_raw = np.full(n_bins, np.nan)
    if valid_z.sum() >= 3:
        grad_filled = np.gradient(z_filled, bin_size)
        dzdx_raw[valid_z] = grad_filled[valid_z]

    return dict(
        x=x_centers,
        z=z_smooth,
        z_raw=z_bin,
        slope_deg=np.degrees(np.arctan(dzdx_smooth)),
        slope_raw_deg=np.degrees(np.arctan(dzdx_raw)),
        kappa=kappa,
        kappa_abs=np.abs(kappa),
        valid_z_mask=valid_z,
        valid_s_mask=valid_z,
        bin_size=bin_size,
        method=method_label,
    )



# ----------------------------------------------------------
# Method 4: IRLS quadric (global robust surface fit)
# ----------------------------------------------------------
def fit_irls_quadric(pts,
                     max_iter=IRLS_MAX_ITER,
                     tol=IRLS_TOL,
                     c_tukey=IRLS_C_TUKEY):
    """Fit z = a x^2 + b xy + c y^2 + d x + e y + f via IRLS (Tukey biweight).

    Returns
    -------
    coeffs    : (6,) array  -> [a, b, c, d, e, f]
    weights   : (N,) array  -> final per-point weights in [0, 1]
    residuals : (N,) array  -> z - z_fit at convergence
    n_iter    : int         -> iterations actually run
    """
    x, y, z = pts[:, 0], pts[:, 1], pts[:, 2]
    A = np.column_stack([x**2, x*y, y**2, x, y, np.ones_like(x)])

    # Iteration 0: unweighted LS
    coeffs, *_ = np.linalg.lstsq(A, z, rcond=None)

    weights = np.ones_like(z)
    residuals = z - A @ coeffs

    for it in range(max_iter):
        residuals = z - A @ coeffs
        # Robust scale: 1.4826 * MAD matches std for Gaussian data
        scale = 1.4826 * np.median(np.abs(residuals - np.median(residuals)))
        if scale < 1e-12:
            break

        u = residuals / (c_tukey * scale)
        weights = np.where(np.abs(u) < 1, (1 - u**2)**2, 0.0)

        sw = np.sqrt(weights)
        Aw = A * sw[:, None]
        zw = z * sw
        new_coeffs, *_ = np.linalg.lstsq(Aw, zw, rcond=None)

        rel_delta = np.linalg.norm(new_coeffs - coeffs) / (np.linalg.norm(coeffs) + 1e-12)
        coeffs = new_coeffs
        if rel_delta < tol:
            return coeffs, weights, residuals, it + 1

    return coeffs, weights, residuals, max_iter


def compute_road_profile_from_quadric(ground_points, method_label="IRLS"):
    """Forward 1-D profile from a global IRLS-quadric fit.

    Same output dict as the per-bin methods so save_profile() works untouched.
    Slope and curvature are read off the quadric analytically along the slice
    y = IRLS_Y_CENTER:
        z(x)  = a x^2 + (b y0 + d) x + (c y0^2 + e y0 + f)
        z'(x) = 2 a x + b y0 + d
        z''(x)= 2 a                              (constant)
        kappa = z'' / (1 + z'^2)^(3/2)           (signed Frenet)
    """
    x_range      = ROI_X_RANGE
    y_half_width = abs(ROI_Y_RANGE[1])
    bin_size     = PROFILE_BIN_SIZE
    min_pts      = PROFILE_MIN_PTS
    y0           = IRLS_Y_CENTER

    # 1. Same ROI crop as the other three methods.
    m = ((ground_points[:, 0] >= x_range[0]) &
         (ground_points[:, 0] <= x_range[1]) &
         (np.abs(ground_points[:, 1]) <= y_half_width))
    pts = ground_points[m]
    if len(pts) < 10:
        print(f"  [{method_label}] Not enough ground points in the forward strip.")
        return None

    # 2. Fit one quadric to all points in the strip.
    try:
        coeffs, weights, residuals, n_iter = fit_irls_quadric(pts)
    except Exception as exc:
        print(f"  [{method_label}] IRLS fit failed: {exc}")
        return None
    a, b, c_q, d, e, f = coeffs   # c_q to avoid shadowing module-level names

    # 3. Sample along the same x-bin grid the other methods use.
    x_edges   = np.arange(x_range[0], x_range[1] + bin_size, bin_size)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    n_bins    = len(x_centers)

    # 3a. Analytic z, slope, curvature along the slice y = y0.
    z_smooth    = (a*x_centers**2 + b*x_centers*y0 + c_q*y0**2
                   + d*x_centers + e*y0 + f)
    dzdx_smooth = 2*a*x_centers + b*y0 + d
    d2zdx2      = np.full_like(x_centers, 2*a)
    kappa       = d2zdx2 / (1.0 + dzdx_smooth ** 2) ** 1.5

    # 3b. Raw bin medians (for the orange dots in the road-profile plot).
    z_bin = np.full(n_bins, np.nan)
    for i in range(n_bins):
        in_bin = (pts[:, 0] >= x_edges[i]) & (pts[:, 0] < x_edges[i + 1])
        bin_pts = pts[in_bin]
        if len(bin_pts) < min_pts:
            continue
        z_bin[i] = float(np.median(bin_pts[:, 2]))

    valid_z = ~np.isnan(z_bin)

    # 3c. "Raw" slope = finite difference of the bin medians, matching what
    #     compute_road_profile_from_heights() stores. Honest NaN where z is missing.
    dzdx_raw = np.full(n_bins, np.nan)
    if valid_z.sum() >= 3:
        z_filled    = np.interp(x_centers, x_centers[valid_z], z_bin[valid_z])
        grad_filled = np.gradient(z_filled, bin_size)
        dzdx_raw[valid_z] = grad_filled[valid_z]

    return dict(
        x=x_centers,
        z=z_smooth,
        z_raw=z_bin,
        slope_deg=np.degrees(np.arctan(dzdx_smooth)),
        slope_raw_deg=np.degrees(np.arctan(dzdx_raw)),
        kappa=kappa,
        kappa_abs=np.abs(kappa),
        valid_z_mask=valid_z,
        valid_s_mask=np.ones(n_bins, dtype=bool),   # quadric defined everywhere
        bin_size=bin_size,
        method=method_label,
    )


In [ ]:
def save_profile(profile, filter_label, ts_sec, results_dir=RESULTS_DIR):
    """Save a profile dict as compressed .npz.
    Saved to: results_dir / YYYY_MM_DD / HH_MM_SS / ts_int / {method}_{filter_label}_{unix_ms}.npz
        e.g. D:\\Validation_results\\2026_05_22\\10_40_00\\1779439338\\PCA_patchwork_1779439338625.npz
    """
    if profile is None:
        print(f"  Skipping save: profile is None ({filter_label}).")
        return

    from datetime import datetime

    method   = profile["method"]
    stem     = timestamp_stem(ts_sec)
    dt       = datetime.fromtimestamp(ts_sec)
    date_str = dt.strftime("%Y_%m_%d")
    time_str = dt.strftime("%H_%M_%S")
    ts_int   = str(int(ts_sec))

    subdir = os.path.join(results_dir, date_str, time_str, ts_int)
    os.makedirs(subdir, exist_ok=True)

    fname  = f"{method}_{filter_label}_{stem}.npz"
    fpath  = os.path.join(subdir, fname)

    np.savez_compressed(
        fpath,
        x             = profile["x"],
        z             = profile["z"],
        s             = profile["x"],
        #s             = np.linspace(ROI_X_RANGE[0], ROI_X_RANGE[1],
        #                            int((ROI_X_RANGE[1] - ROI_X_RANGE[0]) / PROFILE_BIN_SIZE) + 1),
        t             = np.array([ts_sec]),
        method        = np.array([f"{method}_{filter_label}"]),
        filter_label  = np.array([filter_label]),
        slope_raw_deg = profile["slope_raw_deg"],
        slope_deg     = profile["slope_deg"],
        z_raw         = profile["z_raw"],
        kappa         = profile["kappa"],
        kappa_abs     = profile["kappa_abs"],
        bin_size      = np.array([profile["bin_size"]]),
        valid_z_mask  = profile["valid_z_mask"],
        valid_s_mask  = profile["valid_s_mask"],
    )
    print(f"  saved {method:<14s} -> {fname}")


## 8. Stage 2 driver — four methods × two filters × every timestamp

In [ ]:
for k, (ts, pw_path, csf_path) in enumerate(STAGE1_OUTPUTS, start=1):
    print("=" * 64)
    print(f"[{k}/{len(STAGE1_OUTPUTS)}]  Stage 2  --  t = {ts:.6f} s")
    print("=" * 64)

    for filter_label, bin_path in (("patchwork", pw_path), ("csf", csf_path)):
        print(f"  [{filter_label}]  {os.path.basename(bin_path)}")
        ground = load_nuscenes_bin(bin_path)
        print(f"    loaded {len(ground):,} ground points")

        prof_pca = compute_road_profile_from_normals(ground, fit_pca_normal,    "PCA")
        prof_ran = compute_road_profile_from_normals(ground, fit_ransac_normal, "RANSAC")
        prof_hgt = compute_road_profile_from_heights(ground)

        save_profile(prof_pca, filter_label, ts)
        save_profile(prof_ran, filter_label, ts)
        save_profile(prof_hgt, filter_label, ts)

print(f"\nStage 2 complete: wrote {6 * len(STAGE1_OUTPUTS)} .npz file(s) to {RESULTS_DIR}")